# GraphRetro 数据处理教程

## 概述

本 Notebook 展示 GraphRetro 的完整数据处理管线。GraphRetro 的数据处理可以分为以下 **6 个步骤**：

```
Step 1: 产物 SMILES 标准化 (Canonicalization)
          │
Step 2: 反应核心提取 (Reaction Core Extraction)
          │
Step 3: 键编辑与离去基解析 (Edit & LG Parsing)
          │
Step 4: 编辑应用与中间体生成 (Apply Edits → Synthons)
          │
Step 5: 分子图特征构建 (Graph Feature Construction)
          │
Step 6: 训练张量打包 (Tensor Batching)
```

我们将使用一个包含 20 条反应的**微型教学数据集**来完整展示每一步。

In [1]:
# ============================================================
# 基础导入与路径设置
# ============================================================
import os, sys
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw, AllChem
from collections import defaultdict, namedtuple, deque
from typing import List, Dict, Tuple, Set
import copy

def find_project_root(marker='source_repos'):
    path = os.path.abspath('.')
    while path != '/':
        if os.path.isdir(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f'找不到包含 {marker} 的项目根目录')

PROJECT_ROOT = find_project_root()
SOURCE_DIR = os.path.join(PROJECT_ROOT, 'source_repos', 'graphretro')
TUTORIAL_DIR = os.path.join(PROJECT_ROOT, 'teaching_demos', '2.single_step_retro_tutorial',
                            '2.3.semi-template', '2.3.2.graphretro')
sys.path.insert(0, SOURCE_DIR)

print(f'项目根目录: {PROJECT_ROOT}')
print(f'源码目录:   {os.path.relpath(SOURCE_DIR, PROJECT_ROOT)}')

项目根目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis
源码目录:   source_repos/graphretro


## Step 0: 加载教学数据集

我们使用的数据来源于 **USPTO-50K** 数据集，每条记录包含：
- `id`: 专利编号
- `class`: 反应类别（1-10，对应不同反应类型）
- `reactants>reagents>production`: 带原子映射的反应 SMILES

> **原子映射 (Atom Mapping)**：用 `:n` 标记原子编号，使我们可以追踪产物和反应物中对应的原子。
> 例如 `[CH3:1][OH:2]` 表示碳原子编号为 1，氧原子编号为 2。

In [2]:
# 加载微型教学数据集
data_path = os.path.join(TUTORIAL_DIR, 'data', 'demo_data.csv')
df = pd.read_csv(data_path)
print(f'加载了 {len(df)} 条反应')
print(f'列名: {list(df.columns)}')
print()

# 展示前 3 条数据
for i in range(min(3, len(df))):
    row = df.iloc[i]
    rxn_col = 'reactants>reagents>production'
    rxn_smi = row[rxn_col]
    parts = rxn_smi.split('>>')
    if len(parts) == 2:
        reactants, product = parts
    else:
        # 处理 reactants>reagents>production 格式
        r_parts = rxn_smi.split('>')
        reactants = r_parts[0]
        product = r_parts[-1]
    
    print(f'反应 {i+1} (类别 {row["class"]})')
    print(f'  反应物: {reactants[:80]}...' if len(reactants) > 80 else f'  反应物: {reactants}')
    print(f'  产物:   {product[:80]}...' if len(product) > 80 else f'  产物:   {product}')
    print()

加载了 20 条反应
列名: ['id', 'class', 'reactants>reagents>production']

反应 1 (类别 5)
  反应物: [C:12](=[O:13])([O:14][C:15]([CH3:16])([CH3:17])[CH3:18])[O:27][C:25]([O:24][C:2...
  产物:   [CH3:1][C:2](=[O:3])[c:4]1[cH:5][cH:6][c:7]2[c:8]([cH:9][cH:10][n:11]2[C:12](=[O...

反应 2 (类别 5)
  反应物: [C:13](=[O:14])([O:15][C:16]([CH3:17])([CH3:18])[CH3:19])[O:35][C:33]([O:32][C:2...
  产物:   [CH3:1][c:2]1[cH:3][cH:4][c:5]([S:6](=[O:7])(=[O:8])[O:9][C@@H:10]2[CH2:11][N:12...

反应 3 (类别 10)
  反应物: [Br:28][N:35]1[C:30](=[O:29])[CH2:31][CH2:32][C:33]1=[O:34].[CH3:1][CH2:2][O:3][...
  产物:   [CH3:1][CH2:2][O:3][C:4](=[O:5])[c:6]1[n:7][n:8](-[c:9]2[cH:10][cH:11][c:12]([Cl...



## Step 1: 产物 SMILES 标准化

### 为什么需要标准化？

同一个分子可以有多种 SMILES 表示，例如：
- `c1ccccc1` 和 `C1=CC=CC=C1` 都表示苯
- `OCC` 和 `CCO` 都表示乙醇

标准化确保每个分子只有唯一的 SMILES 表示，同时需要重新编号原子映射以匹配标准化后的原子顺序。

### GraphRetro 的标准化流程

```
原始产物 SMILES → 去除立体化学标记 → RDKit 标准化 → 重新编号原子映射 → 更新反应物映射
```

> 对应源码: `data_process/canonicalize_prod.py` 中的 `canonicalize_prod()` 和 `remap_rxn_smi()`

In [3]:
# ============================================================
# Step 1: 产物标准化
# 对应源码: data_process/canonicalize_prod.py
# ============================================================

def canonicalize_prod(p: str):
    """
    标准化产物 SMILES：
    1. 解析产物分子
    2. 移除原子映射
    3. 生成标准化 SMILES
    4. 恢复原子映射（新的编号顺序）
    
    返回: (标准化SMILES, 旧编号→新编号映射)
    """
    prd = Chem.MolFromSmiles(p)
    if prd is None:
        return None, {}
    
    # 记录原始原子映射
    old_amap = {atom.GetIdx(): atom.GetAtomMapNum() for atom in prd.GetAtoms()}
    
    # 移除映射得到标准化 SMILES
    for atom in prd.GetAtoms():
        atom.SetAtomMapNum(0)
    cano_smi = Chem.MolToSmiles(prd)
    
    # 重新解析标准化分子
    cano_mol = Chem.MolFromSmiles(cano_smi)
    if cano_mol is None:
        return None, {}
    
    # 建立标准化后的原子编号映射
    # 通过子结构匹配来对齐原子
    match = prd.GetSubstructMatch(cano_mol)
    if not match:
        # 如果子结构匹配失败，用简单的顺序映射
        amap_dict = {old_amap[i]: i + 1 for i in range(prd.GetNumAtoms())}
    else:
        amap_dict = {}
        for new_idx, old_idx in enumerate(match):
            amap_dict[old_amap[old_idx]] = new_idx + 1
    
    # 给标准化分子加上新的原子映射
    for atom in cano_mol.GetAtoms():
        atom.SetAtomMapNum(atom.GetIdx() + 1)
    
    result = Chem.MolToSmiles(cano_mol)
    return result, amap_dict


def remap_rxn_smi(rxn_smi: str):
    """
    标准化完整反应 SMILES：
    1. 标准化产物
    2. 根据产物的新映射重编号反应物
    """
    r, p = rxn_smi.split('>>')
    
    # 标准化产物
    cano_p, amap_dict = canonicalize_prod(p)
    if cano_p is None:
        return None
    
    # 重映射反应物中的原子编号
    reac_mol = Chem.MolFromSmiles(r)
    if reac_mol is None:
        return None
    
    for atom in reac_mol.GetAtoms():
        old_map = atom.GetAtomMapNum()
        if old_map in amap_dict:
            atom.SetAtomMapNum(amap_dict[old_map])
        # 映射不在产物中的原子保持不变（离去基或试剂）
    
    cano_r = Chem.MolToSmiles(reac_mol)
    return f'{cano_r}>>{cano_p}'


# === 演示标准化过程 ===
print('=== Step 1: 产物 SMILES 标准化演示 ===')
print()

# 用第一条反应做演示
rxn_col = 'reactants>reagents>production'
sample_rxn = df.iloc[0][rxn_col]
r_parts = sample_rxn.split('>')
reactants_smi = r_parts[0]
product_smi = r_parts[-1]

print(f'原始产物: {product_smi[:100]}...')
cano_p, amap = canonicalize_prod(product_smi)
print(f'标准化产物: {cano_p[:100]}...' if cano_p else '标准化失败')
print(f'映射关系 (旧→新): {dict(list(amap.items())[:5])}...')

=== Step 1: 产物 SMILES 标准化演示 ===

原始产物: [CH3:1][C:2](=[O:3])[c:4]1[cH:5][cH:6][c:7]2[c:8]([cH:9][cH:10][n:11]2[C:12](=[O:13])[O:14][C:15]([C...
标准化产物: [CH3:1][C:2](=[O:3])[c:4]1[cH:5][cH:6][c:7]2[c:8]([cH:9][cH:10][n:11]2[C:12](=[O:13])[O:14][C:15]([C...
映射关系 (旧→新): {1: 1, 2: 2, 3: 3, 4: 4, 5: 5}...


## Step 2: 反应核心提取 (Reaction Core)

### 什么是反应核心？

反应核心（Reaction Core）是指反应中**发生键变化的原子集合**。GraphRetro 通过比较产物和反应物中相同原子映射之间的键差异来提取反应核心。

### 键编辑（Core Edits）的格式

每个编辑以字符串 `"a1:a2:b_prod:b_reac"` 表示：
- `a1, a2`: 两个原子的映射编号
- `b_prod`: 产物中的键序（0.0=无键, 1.0=单键, 1.5=芳香键, 2.0=双键, 3.0=三键）
- `b_reac`: 反应物中的键序

例如 `"3:5:1.0:0.0"` 表示原子3和原子5之间从单键变为无键（即断键）。

> 对应源码: `seq_graph_retro/utils/parse.py` 中的 `get_reaction_core()`

In [4]:
# ============================================================
# Step 2: 反应核心提取
# 对应源码: seq_graph_retro/utils/parse.py → get_reaction_core()
# ============================================================

def get_bond_info(mol: Chem.Mol) -> Dict:
    """
    提取分子中所有键的信息。
    返回: {(原子映射1, 原子映射2): [键序, 键索引]} 的字典
    """
    if mol is None:
        return {}
    bond_info = {}
    for bond in mol.GetBonds():
        a_start = bond.GetBeginAtom().GetAtomMapNum()
        a_end = bond.GetEndAtom().GetAtomMapNum()
        key_pair = tuple(sorted([a_start, a_end]))
        bond_info[key_pair] = [bond.GetBondTypeAsDouble(), bond.GetIdx()]
    return bond_info


def get_reaction_core(r: str, p: str, kekulize: bool = False, 
                      use_h_labels: bool = False) -> Tuple[Set, List]:
    """
    提取反应核心：比较产物和反应物的键差异。
    
    参数:
        r: 反应物 SMILES
        p: 产物 SMILES
        kekulize: 是否进行 Kekulize 处理
        use_h_labels: 是否包含氢原子变化
    
    返回:
        rxn_core: 反应核心原子映射编号集合
        core_edits: 键编辑列表 ["a1:a2:b_prod:b_reac", ...]
    """
    reac_mol = Chem.MolFromSmiles(r)
    prod_mol = Chem.MolFromSmiles(p)
    
    if reac_mol is None or prod_mol is None:
        return set(), []
    
    # 给没有映射的反应物原子分配映射编号
    max_amap = max([atom.GetAtomMapNum() for atom in reac_mol.GetAtoms()])
    for atom in reac_mol.GetAtoms():
        if atom.GetAtomMapNum() == 0:
            atom.SetAtomMapNum(max_amap + 1)
            max_amap += 1
    
    prod_bonds = get_bond_info(prod_mol)
    reac_bonds = get_bond_info(reac_mol)
    p_amap_idx = {atom.GetAtomMapNum(): atom.GetIdx() for atom in prod_mol.GetAtoms()}
    reac_amap = {atom.GetAtomMapNum(): atom.GetIdx() for atom in reac_mol.GetAtoms()}
    
    rxn_core = set()
    core_edits = []
    
    # 情况 1: 产物中有键，但反应物中键序不同
    for bond in prod_bonds:
        if bond in reac_bonds and prod_bonds[bond][0] != reac_bonds[bond][0]:
            a_start, a_end = sorted(bond)
            edit = f"{a_start}:{a_end}:{prod_bonds[bond][0]}:{reac_bonds[bond][0]}"
            core_edits.append(edit)
            rxn_core.update([a_start, a_end])
        
        # 情况 2: 产物中有键，反应物中没有（需要断键）
        if bond not in reac_bonds:
            a_start, a_end = sorted(bond)
            edit = f"{a_start}:{a_end}:{prod_bonds[bond][0]}:{0.0}"
            core_edits.append(edit)
            rxn_core.update([a_start, a_end])
    
    # 情况 3: 反应物中有键，产物中没有（需要成键）
    for bond in reac_bonds:
        if bond not in prod_bonds:
            amap1, amap2 = bond
            if (amap1 in p_amap_idx) and (amap2 in p_amap_idx):
                a_start, a_end = sorted([amap1, amap2])
                edit = f"{a_start}:{a_end}:{0.0}:{reac_bonds[bond][0]}"
                core_edits.append(edit)
                rxn_core.update([a_start, a_end])
    
    # 情况 4: 仅氢原子数发生变化
    if use_h_labels and len(rxn_core) == 0:
        for atom in prod_mol.GetAtoms():
            amap_num = atom.GetAtomMapNum()
            numHs_prod = atom.GetTotalNumHs()
            if amap_num in reac_amap:
                numHs_reac = reac_mol.GetAtomWithIdx(reac_amap[amap_num]).GetTotalNumHs()
                if numHs_prod != numHs_reac:
                    edit = f"{amap_num}:{0}:{1.0}:{0.0}"
                    core_edits.append(edit)
                    rxn_core.add(amap_num)
    
    return rxn_core, core_edits


# === 演示反应核心提取 ===
print('=== Step 2: 反应核心提取演示 ===')
print()

# 使用数据集中的反应
sample_rxn = df.iloc[0][rxn_col]
r_parts = sample_rxn.split('>')
r_smi = r_parts[0]
p_smi = r_parts[-1]

rxn_core, core_edits = get_reaction_core(r_smi, p_smi, kekulize=False, use_h_labels=True)

print(f'反应物: {r_smi[:80]}...' if len(r_smi) > 80 else f'反应物: {r_smi}')
print(f'产物:   {p_smi[:80]}...' if len(p_smi) > 80 else f'产物:   {p_smi}')
print(f'\n反应核心原子: {rxn_core}')
print(f'键编辑数: {len(core_edits)}')
for edit in core_edits:
    a1, a2, b_prod, b_reac = edit.split(':')
    b_prod, b_reac = float(b_prod), float(b_reac)
    if b_reac == 0.0:
        change_type = '断键'
    elif b_prod == 0.0:
        change_type = '成键'
    else:
        change_type = '键序变化'
    print(f'  编辑: 原子{a1}-原子{a2}: {b_prod}→{b_reac} ({change_type})')

=== Step 2: 反应核心提取演示 ===

反应物: [C:12](=[O:13])([O:14][C:15]([CH3:16])([CH3:17])[CH3:18])[O:27][C:25]([O:24][C:2...
产物:   [CH3:1][C:2](=[O:3])[c:4]1[cH:5][cH:6][c:7]2[c:8]([cH:9][cH:10][n:11]2[C:12](=[O...

反应核心原子: {11, 12}
键编辑数: 1
  编辑: 原子11-原子12: 1.0→0.0 (断键)


In [5]:
# 批量处理前 5 条反应，展示不同反应类型的编辑模式
print('=== 批量反应核心提取 ===')
print(f'{"#":>3} {"类别":>4} {"核心原子数":>8} {"编辑数":>6} {"编辑详情"}')
print('-' * 70)

for i in range(min(10, len(df))):
    row = df.iloc[i]
    rxn_smi = row[rxn_col]
    r_parts = rxn_smi.split('>')
    r_smi = r_parts[0]
    p_smi = r_parts[-1]
    
    core, edits = get_reaction_core(r_smi, p_smi, kekulize=False, use_h_labels=True)
    edits_str = '; '.join(edits) if edits else 'N/A'
    if len(edits_str) > 40:
        edits_str = edits_str[:40] + '...'
    print(f'{i+1:3d} {row["class"]:>4} {len(core):>8d} {len(edits):>6d} {edits_str}')

=== 批量反应核心提取 ===
  #   类别    核心原子数    编辑数 编辑详情
----------------------------------------------------------------------
  1    5        2      1 11:12:1.0:0.0
  2    5        2      1 12:13:1.0:0.0
  3   10        2      1 27:28:1.0:0.0
  4    5        2      1 6:8:1.0:0.0
  5    5        2      1 2:3:1.0:0.0
  6   10        2      1 10:11:1.0:0.0
  7    8        2      1 18:19:2.0:0.0
  8    5        2      1 6:8:1.0:0.0
  9   10        2      1 7:8:1.0:0.0
 10    8        2      1 9:10:2.0:1.0


## Step 3: 完整反应信息解析（键编辑 + 离去基）

### 离去基（Leaving Groups）是什么？

在逆合成中，将产物断键后得到的中间体称为**合成子（synthons）**。但合成子不一定是完整的反应物——反应物中可能包含一些**离去基团**，这些基团在正向反应中会脱离。

GraphRetro 使用 BFS（广度优先搜索）从反应核心出发，遍历反应物中不在产物中的原子，提取离去基信息。

### ReactionInfo 数据结构

```python
ReactionInfo = namedtuple('ReactionInfo', [
    'rxn_smi',      # 完整反应 SMILES
    'core',         # 反应核心原子集合
    'core_edits',   # 键编辑列表 ["a1:a2:b1:b2", ...]
    'lg_edits',     # 离去基编辑列表
    'attach_atoms',  # 离去基连接点
    'rxn_class'     # 反应类别
])
```

> 对应源码: `seq_graph_retro/utils/parse.py` → `get_reaction_info()`

In [6]:
# ============================================================
# Step 3: 完整反应信息解析（包含离去基）
# 对应源码: seq_graph_retro/utils/parse.py → get_reaction_info()
# ============================================================

ReactionInfo = namedtuple('ReactionInfo', 
    ['rxn_smi', 'core', 'core_edits', 'lg_edits', 'attach_atoms', 'rxn_class'])


def get_reaction_info(rxn_smi: str, kekulize: bool = False, 
                      use_h_labels: bool = False, rxn_class: int = None) -> ReactionInfo:
    """
    解析完整的反应信息，包括:
    1. 反应核心和键编辑
    2. 离去基编辑
    3. 附着原子（离去基连接点）
    """
    r, p = rxn_smi.split('>>')
    reac_mol = Chem.MolFromSmiles(r)
    prod_mol = Chem.MolFromSmiles(p)
    
    if reac_mol is None or prod_mol is None:
        return None
    
    # 第一步：提取反应核心
    rxn_core, core_edits = get_reaction_core(r, p, kekulize=kekulize, use_h_labels=use_h_labels)
    
    prod_bonds = get_bond_info(prod_mol)
    p_amap_idx = {atom.GetAtomMapNum(): atom.GetIdx() for atom in prod_mol.GetAtoms()}
    
    # 给未映射的反应物原子分配编号
    max_amap = max([atom.GetAtomMapNum() for atom in reac_mol.GetAtoms()])
    for atom in reac_mol.GetAtoms():
        if atom.GetAtomMapNum() == 0:
            atom.SetAtomMapNum(max_amap + 1)
            max_amap += 1
    
    reac_amap = {atom.GetAtomMapNum(): atom.GetIdx() for atom in reac_mol.GetAtoms()}
    
    # 标记产物中已存在且非核心的原子为"已访问"
    visited = set([atom.GetIdx() for atom in reac_mol.GetAtoms()
                   if (atom.GetAtomMapNum() in p_amap_idx) and 
                      (atom.GetAtomMapNum() not in rxn_core)])
    
    # 第二步：BFS 提取离去基编辑
    lg_edits = []
    for atom_map in rxn_core:
        if atom_map not in reac_amap:
            continue
        root = reac_mol.GetAtomWithIdx(reac_amap[atom_map])
        queue = deque([root])
        
        while len(queue) > 0:
            x = queue.popleft()
            neis = sorted(x.GetNeighbors(), key=lambda a: a.GetIdx())
            
            for y in neis:
                y_idx = y.GetIdx()
                if y_idx in visited:
                    continue
                
                # 检测此键是否需要记录
                bo = reac_mol.GetBondBetweenAtoms(x.GetIdx(), y_idx).GetBondTypeAsDouble()
                amap1, amap2 = sorted([y.GetAtomMapNum(), x.GetAtomMapNum()])
                bond = (amap1, amap2)
                
                if bond in prod_bonds and prod_bonds[bond][0] == bo:
                    pass  # 无变化
                elif bond in prod_bonds and prod_bonds[bond][0] != bo:
                    bo_old = prod_bonds[bond][0]
                    edit = f"{amap1}:{amap2}:{bo_old}:{bo}"
                    if edit not in lg_edits and edit not in core_edits:
                        lg_edits.append(edit)
                else:
                    edit = f"{amap1}:{amap2}:{0.0}:{bo}"
                    if edit not in lg_edits and edit not in core_edits:
                        lg_edits.append(edit)
                
                visited.add(y_idx)
                queue.append(y)
    
    # 简化的附着原子提取
    attach_atoms = []
    
    r_new = Chem.MolToSmiles(reac_mol)
    p_new = Chem.MolToSmiles(prod_mol)
    rxn_smi_new = f"{r_new}>>{p_new}"
    
    return ReactionInfo(
        rxn_smi=rxn_smi_new, core=rxn_core, core_edits=core_edits,
        lg_edits=lg_edits, attach_atoms=attach_atoms, rxn_class=rxn_class
    )


# === 演示完整反应信息解析 ===
print('=== Step 3: 完整反应信息解析演示 ===')
print()

# 解析所有反应
all_infos = []
for i in range(len(df)):
    row = df.iloc[i]
    rxn_smi = row[rxn_col]
    r_parts = rxn_smi.split('>')
    r_smi = r_parts[0]
    p_smi = r_parts[-1]
    
    # 只保留有原子映射的部分
    rxn_formatted = f"{r_smi}>>{p_smi}"
    info = get_reaction_info(rxn_formatted, kekulize=False, use_h_labels=True, 
                            rxn_class=row['class'] if 'class' in row.index else None)
    if info is not None:
        all_infos.append(info)

print(f'成功解析 {len(all_infos)}/{len(df)} 条反应')
print()

# 展示前 3 条的详细信息
for i, info in enumerate(all_infos[:3]):
    print(f'--- 反应 {i+1} ---')
    print(f'  反应核心原子: {info.core}')
    print(f'  核心编辑 ({len(info.core_edits)}):')
    for edit in info.core_edits:
        print(f'    {edit}')
    print(f'  离去基编辑 ({len(info.lg_edits)}):')
    for edit in info.lg_edits[:3]:
        print(f'    {edit}')
    if len(info.lg_edits) > 3:
        print(f'    ... 共 {len(info.lg_edits)} 个')
    print()

=== Step 3: 完整反应信息解析演示 ===

成功解析 20/20 条反应

--- 反应 1 ---
  反应核心原子: {11, 12}
  核心编辑 (1):
    11:12:1.0:0.0
  离去基编辑 (8):
    12:27:0.0:1.0
    25:27:0.0:1.0
    24:25:0.0:1.0
    ... 共 8 个

--- 反应 2 ---
  反应核心原子: {12, 13}
  核心编辑 (1):
    12:13:1.0:0.0
  离去基编辑 (8):
    13:35:0.0:1.0
    33:35:0.0:1.0
    32:33:0.0:1.0
    ... 共 8 个

--- 反应 3 ---
  反应核心原子: {27, 28}
  核心编辑 (1):
    27:28:1.0:0.0
  离去基编辑 (7):
    28:35:0.0:1.0
    30:35:0.0:1.0
    33:35:0.0:1.0
    ... 共 7 个



## Step 4: 编辑应用与中间体生成

### 从产物到中间体（Synthons）

通过将核心编辑应用到产物分子上，我们可以得到中间体（synthons）：

```
产物 + 编辑 → 中间体（可能是多个断开的分子片段）
```

例如，断开一个 C-N 键：
```
PhCONHR → PhCO· + ·NHR  (两个合成子)
```

> 对应源码: `seq_graph_retro/utils/chem.py` → `apply_edits_to_mol()`

In [7]:
# ============================================================
# Step 4: 编辑应用与中间体生成
# 对应源码: seq_graph_retro/utils/chem.py → apply_edits_to_mol()
# ============================================================

BOND_FLOAT_TO_TYPE = {
    0.0: None,
    1.0: Chem.rdchem.BondType.SINGLE,
    2.0: Chem.rdchem.BondType.DOUBLE,
    3.0: Chem.rdchem.BondType.TRIPLE,
    1.5: Chem.rdchem.BondType.AROMATIC,
}

def apply_edits_to_mol(mol: Chem.Mol, edits: List[str]) -> Chem.Mol:
    """
    将编辑应用到分子上。
    
    参数:
        mol: 产物分子
        edits: 编辑列表 ["a1:a2:b_new:b_old", ...]
              注意: 这里 b_new 是编辑后的键序（即反应物中的键序）
    
    返回:
        编辑后的分子（中间体/合成子）
    """
    new_mol = Chem.RWMol(mol)
    amap_idx = {atom.GetAtomMapNum(): atom.GetIdx() for atom in new_mol.GetAtoms()}
    
    for edit in edits:
        a1, a2, b_prod, b_reac = edit.split(':')
        a1, a2 = int(a1), int(a2)
        b_reac = float(b_reac)  # 目标键序
        
        if a2 == 0:
            # 氢变化编辑，跳过键操作
            continue
        
        if a1 not in amap_idx or a2 not in amap_idx:
            continue
        
        idx1 = amap_idx[a1]
        idx2 = amap_idx[a2]
        
        # 先移除旧键
        old_bond = new_mol.GetBondBetweenAtoms(idx1, idx2)
        if old_bond is not None:
            new_mol.RemoveBond(idx1, idx2)
        
        # 如果目标键序不为 0，添加新键
        if b_reac != 0.0 and BOND_FLOAT_TO_TYPE.get(b_reac) is not None:
            new_mol.AddBond(idx1, idx2, BOND_FLOAT_TO_TYPE[b_reac])
    
    try:
        Chem.SanitizeMol(new_mol)
        return new_mol.GetMol()
    except:
        return new_mol.GetMol()


def get_fragments(mol: Chem.Mol) -> List[Chem.Mol]:
    """将分子拆分为独立的连通片段"""
    frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
    return list(frags)


# === 演示编辑应用 ===
print('=== Step 4: 编辑应用与中间体生成 ===')
print()

for i, info in enumerate(all_infos[:5]):
    rxn_smi = info.rxn_smi
    r_smi, p_smi = rxn_smi.split('>>')
    prod_mol = Chem.MolFromSmiles(p_smi)
    
    if prod_mol is None or len(info.core_edits) == 0:
        continue
    
    # 应用核心编辑
    frag_mol = apply_edits_to_mol(prod_mol, info.core_edits)
    frags = get_fragments(frag_mol)
    
    print(f'反应 {i+1}:')
    print(f'  产物: {p_smi[:60]}...' if len(p_smi) > 60 else f'  产物: {p_smi}')
    print(f'  编辑: {info.core_edits}')
    print(f'  中间体数量: {len(frags)}')
    for j, frag in enumerate(frags):
        frag_smi = Chem.MolToSmiles(frag)
        print(f'    合成子 {j+1}: {frag_smi}')
    print()

=== Step 4: 编辑应用与中间体生成 ===

反应 1:
  产物: [CH3:1][C:2](=[O:3])[c:4]1[cH:5][cH:6][c:7]2[c:8]([cH:9][cH:...
  编辑: ['11:12:1.0:0.0']
  中间体数量: 2
    合成子 1: [CH3:1][C:2](=[O:3])[c:4]1[cH:5][cH:6][c:7]2[c:8]([cH:9][cH:10][n:11]2)[cH:19]1
    合成子 2: [C:12](=[O:13])[O:14][C:15]([CH3:16])([CH3:17])[CH3:18]

反应 2:
  产物: [CH3:1][c:2]1[cH:3][cH:4][c:5]([S:6](=[O:7])(=[O:8])[O:9][C@...
  编辑: ['12:13:1.0:0.0']
  中间体数量: 2
    合成子 1: [CH3:1][c:2]1[cH:3][cH:4][c:5]([S:6](=[O:7])(=[O:8])[O:9][C@@H:10]2[CH2:11][N:12][C@H:20]3[C@@H:21]2[O:22][CH2:23][C@@H:24]3[OH:25])[cH:26][cH:27]1
    合成子 2: [C:13](=[O:14])[O:15][C:16]([CH3:17])([CH3:18])[CH3:19]

反应 3:
  产物: [CH3:1][CH2:2][O:3][C:4](=[O:5])[c:6]1[n:7][n:8](-[c:9]2[cH:...
  编辑: ['27:28:1.0:0.0']
  中间体数量: 2
    合成子 1: [CH3:1][CH2:2][O:3][C:4](=[O:5])[c:6]1[n:7][n:8](-[c:9]2[cH:10][cH:11][c:12]([Cl:13])[cH:14][c:15]2[Cl:16])[c:17](-[c:18]2[cH:19][cH:20][c:21]([O:22][CH3:23])[cH:24][cH:25]2)[c:26]1[CH2:27]
    合成子 2: [Br:28]

反应 4:
  产物: [CH3:1][C:2]([CH3:3]

[12:03:11] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 7 8 9 10 18


## Step 5: 分子图特征构建

### GraphRetro 的分子图表示

GraphRetro 将分子表示为图 $G = (V, E)$，其中：
- **节点 $V$**：原子，每个原子具有 $d_a$ 维特征向量
- **边 $E$**：化学键，每个键具有 $d_b$ 维特征向量

### 原子特征（ATOM_FDIM）

| 特征 | 维度 | 说明 |
|------|------|------|
| 原子类型 | 65 | One-hot 编码 (C, N, O, ..., unk) |
| 度数 | 10 | 0-9 的 One-hot |
| 形式电荷 | 5 | [-2,-1,0,1,2] 的 One-hot |
| 杂化类型 | 5 | SP/SP2/SP3/SP3D/SP3D2 |
| 总化合价 | 7 | 0-6 的 One-hot |
| 氢原子数 | 5 | [0,1,3,4,5] 的 One-hot |
| 芳香性 | 1 | 布尔值 |
| **总计** | **98** | |

### 键特征（BOND_FDIM = 6）

| 特征 | 维度 | 说明 |
|------|------|------|
| 键类型 | 4 | 单键/双键/三键/芳香键 |
| 共轭 | 1 | 是否共轭 |
| 环内 | 1 | 是否在环内 |

### 图数据结构

GraphRetro 使用两种图表示：
- **有向图**（`G_dir`）：用于消息传递编码器（MPNN），消息沿边方向流动
- **无向图**（`G_undir`）：用于 WLN 编码器，双向邻域信息

> 对应源码: `seq_graph_retro/molgraph/mol_features.py`

In [8]:
# ============================================================
# Step 5: 分子图特征构建
# 对应源码: seq_graph_retro/molgraph/mol_features.py
# ============================================================

# 原子允许集合
ATOM_LIST = ['C', 'N', 'O', 'S', 'F', 'Si', 'P', 'Cl', 'Br', 'Mg', 'Na', 'Ca', 'Fe',
    'As', 'Al', 'I', 'B', 'V', 'K', 'Tl', 'Yb', 'Sb', 'Sn', 'Ag', 'Pd', 'Co', 'Se', 'Ti',
    'Zn', 'H', 'Li', 'Ge', 'Cu', 'Au', 'Ni', 'Cd', 'In', 'Mn', 'Zr', 'Cr', 'Pt', 'Hg', 'Pb',
    'W', 'Ru', 'Nb', 'Re', 'Te', 'Rh', 'Ta', 'Tc', 'Ba', 'Bi', 'Hf', 'Mo', 'U', 'Sm', 'Os', 'Ir',
    'Ce','Gd','Ga','Cs', '*', 'unk']

MAX_NB = 10
DEGREES = list(range(MAX_NB))           # 0-9
HYBRIDIZATION = [Chem.rdchem.HybridizationType.SP,
                 Chem.rdchem.HybridizationType.SP2,
                 Chem.rdchem.HybridizationType.SP3,
                 Chem.rdchem.HybridizationType.SP3D,
                 Chem.rdchem.HybridizationType.SP3D2]
FORMAL_CHARGE = [-1, -2, 1, 2, 0]
VALENCE = [0, 1, 2, 3, 4, 5, 6]
NUM_Hs = [0, 1, 3, 4, 5]

BOND_TYPES = [None, Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
              Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]

ATOM_FDIM = len(ATOM_LIST) + len(DEGREES) + len(FORMAL_CHARGE) + len(HYBRIDIZATION) \
            + len(VALENCE) + len(NUM_Hs) + 1
BOND_FDIM = 6

print(f'原子特征维度 ATOM_FDIM = {ATOM_FDIM}')
print(f'  = {len(ATOM_LIST)} (原子类型) + {len(DEGREES)} (度数) + {len(FORMAL_CHARGE)} (电荷) '
      f'+ {len(HYBRIDIZATION)} (杂化) + {len(VALENCE)} (化合价) + {len(NUM_Hs)} (氢数) + 1 (芳香性)')
print(f'键特征维度 BOND_FDIM = {BOND_FDIM}')

原子特征维度 ATOM_FDIM = 98
  = 65 (原子类型) + 10 (度数) + 5 (电荷) + 5 (杂化) + 7 (化合价) + 5 (氢数) + 1 (芳香性)
键特征维度 BOND_FDIM = 6


In [9]:
# ============================================================
# One-hot 编码函数
# ============================================================

def onek_encoding_unk(x, allowable_set):
    """One-hot 编码，未知元素映射到最后一个位置"""
    if x not in allowable_set:
        x = allowable_set[-1]
    return [float(x == s) for s in allowable_set]


def get_atom_features(atom: Chem.Atom) -> list:
    """
    提取原子特征向量 (ATOM_FDIM 维)
    
    特征组成:
    [原子类型(65) | 度数(10) | 形式电荷(5) | 杂化(5) | 化合价(7) | 氢数(5) | 芳香性(1)]
    """
    return (onek_encoding_unk(atom.GetSymbol(), ATOM_LIST) +
            onek_encoding_unk(atom.GetDegree(), DEGREES) +
            onek_encoding_unk(atom.GetFormalCharge(), FORMAL_CHARGE) +
            onek_encoding_unk(atom.GetHybridization(), HYBRIDIZATION) +
            onek_encoding_unk(atom.GetTotalValence(), VALENCE) +
            onek_encoding_unk(atom.GetTotalNumHs(), NUM_Hs) +
            [float(atom.GetIsAromatic())])


def get_bond_features(bond: Chem.Bond) -> list:
    """
    提取键特征向量 (BOND_FDIM=6 维)
    
    特征组成:
    [单键 | 双键 | 三键 | 芳香键 | 共轭 | 环内]
    """
    bt = bond.GetBondType()
    features = [float(bt == bond_type) for bond_type in BOND_TYPES[1:]]
    features.extend([float(bond.GetIsConjugated()), float(bond.IsInRing())])
    return features


# === 演示特征提取 ===
smi = 'c1ccc(CC(=O)O)cc1'  # 苯乙酸
mol = Chem.MolFromSmiles(smi)

print(f'分子: {smi} ({mol.GetNumAtoms()} 个原子, {mol.GetNumBonds()} 个键)')
print()

# 展示前 3 个原子的特征
print('原子特征示例（前 3 个原子）:')
for i, atom in enumerate(mol.GetAtoms()):
    if i >= 3:
        break
    feat = get_atom_features(atom)
    print(f'  原子 {i} ({atom.GetSymbol()}): 特征维度={len(feat)}')
    
    # 解析各部分
    offset = 0
    print(f'    原子类型: {ATOM_LIST[feat[offset:offset+len(ATOM_LIST)].index(1.0)]}')
    offset += len(ATOM_LIST)
    print(f'    度数: {DEGREES[feat[offset:offset+len(DEGREES)].index(1.0)]}')
    offset += len(DEGREES)
    print(f'    形式电荷: {FORMAL_CHARGE[feat[offset:offset+len(FORMAL_CHARGE)].index(1.0)]}')
    offset += len(FORMAL_CHARGE)
    print(f'    杂化: {["SP","SP2","SP3","SP3D","SP3D2"][feat[offset:offset+len(HYBRIDIZATION)].index(1.0)]}')
    offset += len(HYBRIDIZATION)
    print(f'    化合价: {VALENCE[feat[offset:offset+len(VALENCE)].index(1.0)]}')
    offset += len(VALENCE)
    print(f'    芳香性: {bool(feat[-1])}')
    print()

# 展示前 3 个键的特征
print('键特征示例（前 3 个键）:')
for i, bond in enumerate(mol.GetBonds()):
    if i >= 3:
        break
    feat = get_bond_features(bond)
    bt_names = ['单键', '双键', '三键', '芳香键']
    bt = bt_names[feat[:4].index(1.0)] if 1.0 in feat[:4] else '未知'
    print(f'  键 {i} ({bond.GetBeginAtomIdx()}-{bond.GetEndAtomIdx()}): '
          f'类型={bt}, 共轭={bool(feat[4])}, 环内={bool(feat[5])}')

分子: c1ccc(CC(=O)O)cc1 (10 个原子, 10 个键)

原子特征示例（前 3 个原子）:
  原子 0 (C): 特征维度=98
    原子类型: C
    度数: 2
    形式电荷: 0
    杂化: SP2
    化合价: 4
    芳香性: True

  原子 1 (C): 特征维度=98
    原子类型: C
    度数: 2
    形式电荷: 0
    杂化: SP2
    化合价: 4
    芳香性: True

  原子 2 (C): 特征维度=98
    原子类型: C
    度数: 2
    形式电荷: 0
    杂化: SP2
    化合价: 4
    芳香性: True

键特征示例（前 3 个键）:
  键 0 (0-1): 类型=芳香键, 共轭=True, 环内=True
  键 1 (1-2): 类型=芳香键, 共轭=True, 环内=True
  键 2 (2-3): 类型=芳香键, 共轭=True, 环内=True


In [10]:
# ============================================================
# 分子图邻接表构建
# ============================================================

def get_atom_graph(mol: Chem.Mol) -> np.ndarray:
    """构建原子邻接表: agraph[i] = [邻居原子索引列表]"""
    agraph = np.zeros((mol.GetNumAtoms(), MAX_NB), dtype=np.int32)
    for idx, atom in enumerate(mol.GetAtoms()):
        nei_indices = [nei.GetIdx() + 1 for nei in atom.GetNeighbors()]  # +1 因为 0 是填充
        agraph[idx, :len(nei_indices)] = nei_indices
    return agraph


def get_bond_graph(mol: Chem.Mol) -> np.ndarray:
    """构建键邻接表: bgraph[i] = [与原子i相连的键索引列表]"""
    bgraph = np.zeros((mol.GetNumAtoms(), MAX_NB), dtype=np.int32)
    for idx, atom in enumerate(mol.GetAtoms()):
        bond_indices = []
        for nei in atom.GetNeighbors():
            bond = mol.GetBondBetweenAtoms(atom.GetIdx(), nei.GetIdx())
            bond_indices.append(bond.GetIdx() + 1)  # +1 因为 0 是填充
        bgraph[idx, :len(bond_indices)] = bond_indices
    return bgraph


# === 演示邻接表 ===
smi = 'CCO'  # 乙醇 (简单示例)
mol = Chem.MolFromSmiles(smi)
print(f'分子: {smi}')
print(f'原子: {[atom.GetSymbol() for atom in mol.GetAtoms()]}')
print()

agraph = get_atom_graph(mol)
print('原子邻接表 (agraph):')
for i, atom in enumerate(mol.GetAtoms()):
    neis = agraph[i][agraph[i] > 0] - 1  # 还原为 0-based
    nei_symbols = [mol.GetAtomWithIdx(int(n)).GetSymbol() for n in neis]
    print(f'  原子 {i} ({atom.GetSymbol()}): 邻居 = {nei_symbols}')

bgraph = get_bond_graph(mol)
print('\n键邻接表 (bgraph):')
for i, atom in enumerate(mol.GetAtoms()):
    bonds = bgraph[i][bgraph[i] > 0] - 1  # 还原
    print(f'  原子 {i} ({atom.GetSymbol()}): 连接的键索引 = {list(bonds)}')

分子: CCO
原子: ['C', 'C', 'O']

原子邻接表 (agraph):
  原子 0 (C): 邻居 = ['C']
  原子 1 (C): 邻居 = ['C', 'O']
  原子 2 (O): 邻居 = ['C']

键邻接表 (bgraph):
  原子 0 (C): 连接的键索引 = [0]
  原子 1 (C): 连接的键索引 = [0, 1]
  原子 2 (O): 连接的键索引 = [1]


## Step 5.2: RxnElement 与 BondEditsRxn 图对象

GraphRetro 使用 `RxnElement` 类包装分子及其图属性，`BondEditsRxn` 类进一步添加编辑标签。

### 类层次结构

```
RxnGraph (基类)
  ├── prod_mol: RxnElement    → 单分子图
  ├── frag_mol: MultiElement  → 多片段图（中间体）
  └── reac_mol: MultiElement  → 多片段图（反应物）
  
BondEditsRxn(RxnGraph)
  ├── edit_label: (N_bonds, 5)  → 每个键的目标键序 one-hot
  ├── h_label: (N_atoms,)       → 氢变化标记
  └── done_label: (1,)          → 终止标记
```

> 对应源码: `seq_graph_retro/molgraph/rxn_graphs.py`

In [11]:
# ============================================================
# RxnElement 和 BondEditsRxn 的简化实现
# 对应源码: seq_graph_retro/molgraph/rxn_graphs.py
# ============================================================
import networkx as nx

BOND_FLOATS = [0.0, 1.0, 2.0, 3.0, 1.5]  # 键序: 无键/单键/双键/三键/芳香键

class RxnElement:
    """单分子图表示"""
    def __init__(self, mol: Chem.Mol, rxn_class: int = None):
        self.mol = mol
        self.rxn_class = rxn_class
        self.num_atoms = mol.GetNumAtoms()
        self.num_bonds = mol.GetNumBonds()
        
        # 原子映射
        self.amap_to_idx = {atom.GetAtomMapNum(): atom.GetIdx() 
                           for atom in mol.GetAtoms()}
        self.idx_to_amap = {v: k for k, v in self.amap_to_idx.items()}
        
        # 构建 NetworkX 图
        self.G_undir = nx.Graph(Chem.rdmolops.GetAdjacencyMatrix(mol))
        self.G_dir = nx.DiGraph(Chem.rdmolops.GetAdjacencyMatrix(mol))
        
        for atom in mol.GetAtoms():
            self.G_undir.nodes[atom.GetIdx()]['label'] = atom.GetSymbol()
            self.G_dir.nodes[atom.GetIdx()]['label'] = atom.GetSymbol()
        
        for bond in mol.GetBonds():
            a1, a2 = bond.GetBeginAtom().GetIdx(), bond.GetEndAtom().GetIdx()
            btype = BOND_TYPES.index(bond.GetBondType())
            self.G_undir[a1][a2]['label'] = btype
            self.G_dir[a1][a2]['label'] = btype
            self.G_dir[a2][a1]['label'] = btype
        
        self.atom_scope = (0, self.num_atoms)
        self.bond_scope = (0, self.num_bonds)
    
    def update_atom_scope(self, offset):
        st, le = self.atom_scope
        return (st + offset, le)
    
    def update_bond_scope(self, offset):
        st, le = self.bond_scope
        return (st + offset, le)


class BondEditsRxn:
    """
    带键编辑标签的反应图。
    
    标签:
    - edit_label: (N_bonds, 5) → 每个键的目标键序 [0.0, 1.0, 2.0, 3.0, 1.5]
    - h_label: (N_atoms,) → 氢变化标记
    """
    def __init__(self, prod_mol, frag_mol=None, edits_to_apply=None, rxn_class=None):
        self.prod_mol = RxnElement(prod_mol, rxn_class=rxn_class)
        self.edits_to_apply = edits_to_apply or []
        self.edit_label, self.h_label = self._get_labels()
    
    def _get_labels(self):
        """根据编辑生成训练标签"""
        n_bonds = self.prod_mol.num_bonds
        n_atoms = self.prod_mol.num_atoms
        
        edit_label = np.zeros((n_bonds, len(BOND_FLOATS)))
        h_label = np.zeros(n_atoms)
        
        for edit in self.edits_to_apply:
            a1, a2, b1, b2 = edit.split(':')
            a1, a2 = int(a1), int(a2)
            b2 = float(b2)  # 目标键序
            
            if a2 == 0:
                # 氢变化
                a_start = self.prod_mol.amap_to_idx.get(a1)
                if a_start is not None:
                    h_label[a_start] = 1
            else:
                # 键编辑
                a_start = self.prod_mol.amap_to_idx.get(a1)
                a_end = self.prod_mol.amap_to_idx.get(a2)
                if a_start is not None and a_end is not None:
                    bond = self.prod_mol.mol.GetBondBetweenAtoms(a_start, a_end)
                    if bond is not None:
                        b_idx = bond.GetIdx()
                        edit_label[b_idx][BOND_FLOATS.index(b2)] = 1
        
        return edit_label, h_label


# === 演示 BondEditsRxn ===
print('=== BondEditsRxn 演示 ===')
print()

for i, info in enumerate(all_infos[:3]):
    if len(info.core_edits) == 0:
        continue
    
    rxn_smi = info.rxn_smi
    r_smi, p_smi = rxn_smi.split('>>')
    prod_mol = Chem.MolFromSmiles(p_smi)
    if prod_mol is None:
        continue
    
    rxn_graph = BondEditsRxn(prod_mol, edits_to_apply=info.core_edits)
    
    print(f'反应 {i+1}:')
    print(f'  产物原子数: {rxn_graph.prod_mol.num_atoms}')
    print(f'  产物键数:   {rxn_graph.prod_mol.num_bonds}')
    print(f'  编辑: {info.core_edits}')
    print(f'  edit_label 形状: {rxn_graph.edit_label.shape}')
    print(f'  h_label 形状:    {rxn_graph.h_label.shape}')
    
    # 找出被标记的键
    edited_bonds = np.where(rxn_graph.edit_label.sum(axis=1) > 0)[0]
    for b_idx in edited_bonds:
        target_bo = BOND_FLOATS[np.argmax(rxn_graph.edit_label[b_idx])]
        bond = prod_mol.GetBondWithIdx(int(b_idx))
        print(f'  键 {b_idx} ({bond.GetBeginAtomIdx()}-{bond.GetEndAtomIdx()}): '
              f'当前={bond.GetBondTypeAsDouble()} → 目标={target_bo}')
    
    edited_atoms = np.where(rxn_graph.h_label > 0)[0]
    if len(edited_atoms) > 0:
        print(f'  氢变化原子: {list(edited_atoms)}')
    print()

=== BondEditsRxn 演示 ===

反应 1:
  产物原子数: 19
  产物键数:   20
  编辑: ['11:12:1.0:0.0']
  edit_label 形状: (20, 5)
  h_label 形状:    (19,)
  键 10 (10-11): 当前=1.0 → 目标=0.0

反应 2:
  产物原子数: 27
  产物键数:   29
  编辑: ['12:13:1.0:0.0']
  edit_label 形状: (29, 5)
  h_label 形状:    (27,)
  键 11 (11-12): 当前=1.0 → 目标=0.0

反应 3:
  产物原子数: 28
  产物键数:   30
  编辑: ['27:28:1.0:0.0']
  edit_label 形状: (30, 5)
  h_label 形状:    (28,)
  键 26 (26-27): 当前=1.0 → 目标=0.0



## Step 6: 训练张量打包

### pack_graph_feats() 函数

这是 GraphRetro 数据处理的关键步骤，将多个分子图打包成批处理张量。

**输出张量**:

| 张量 | 形状 | 说明 |
|------|------|------|
| `fnode` | (N_atoms, ATOM_FDIM) | 所有原子的特征矩阵 |
| `fmess` | (N_messages, 2+BOND_FDIM) | 消息特征：[源原子, 目标原子, 键特征] |
| `agraph` | (N_atoms, MAX_NB) | 原子对应的消息索引 |
| `bgraph` | (N_messages, MAX_NB) | 消息的前驱消息索引 |
| `atoms_in_bonds` | (N_bonds, 2) | 每个键连接的两个原子索引 |

**Scope（作用域）**:
- `atom_scope`: 每个分子在 `fnode` 中的起始位置和长度
- `bond_scope`: 每个分子在 `atoms_in_bonds` 中的起始位置和长度

> 对应源码: `seq_graph_retro/data/collate_fns.py` → `pack_graph_feats()`

In [12]:
# ============================================================
# Step 6: 图张量打包（简化版）
# 对应源码: seq_graph_retro/data/collate_fns.py → pack_graph_feats()
# ============================================================
import torch

def create_pad_tensor(alist, pad_val=0):
    """将变长列表填充为等长张量"""
    max_len = max(len(a) for a in alist)
    result = torch.full((len(alist), max_len), pad_val, dtype=torch.long)
    for i, a in enumerate(alist):
        if len(a) > 0:
            result[i, :len(a)] = torch.tensor(a, dtype=torch.long)
    return result


def pack_graph_feats_directed(graph_batch):
    """
    将多个分子图打包为有向消息传递的批处理张量。
    
    消息传递方向: 源原子 → 键 → 目标原子
    
    参数:
        graph_batch: RxnElement 对象列表
    
    返回:
        graph_tensors: (fnode, fmess, agraph, bgraph, atoms_in_bonds)
        scopes: (atom_scope, bond_scope)
    """
    # 初始化填充元素（索引0为填充）
    fnode = [[0.0] * ATOM_FDIM]  # 填充原子，避免对未初始化的 RDKit Atom 提取特征
    fmess = [[0, 0] + [0] * BOND_FDIM]           # 填充消息
    agraph = [[]]                                  # 原子→消息索引
    bgraph = [[]]                                  # 消息→前驱消息索引
    atoms_in_bonds = [[]]                           # 键→原子对
    
    atom_scope, bond_scope = [], []
    edge_dict = {}
    
    for bid, graph in enumerate(graph_batch):
        mol = graph.mol
        atom_offset = len(fnode)
        bond_offset = len(atoms_in_bonds)
        
        atom_scope.append(graph.update_atom_scope(atom_offset))
        bond_scope.append(graph.update_bond_scope(bond_offset))
        
        # 重新编号后的有向图
        G = nx.convert_node_labels_to_integers(graph.G_dir, first_label=atom_offset)
        
        # 原子特征
        fnode.extend([None] * len(G.nodes))
        for v in G.nodes:
            fnode[v] = get_atom_features(mol.GetAtomWithIdx(v - atom_offset))
        agraph.append([])
        
        # 键索引映射
        bond_to_tuple = {bond.GetIdx(): tuple(sorted((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())))
                         for bond in mol.GetBonds()}
        tuple_to_bond = {v: k for k, v in bond_to_tuple.items()}
        
        bond_comp = [None] * mol.GetNumBonds()
        for u, v, attr in G.edges(data='label'):
            bond_feat = get_bond_features(mol.GetBondBetweenAtoms(u - atom_offset, v - atom_offset))
            mess_vec = [u, v] + bond_feat
            
            bond = sorted([u, v])
            if [v, u] not in bond_comp:
                local_bond_key = tuple(sorted([u - atom_offset, v - atom_offset]))
                if local_bond_key in tuple_to_bond:
                    idx = tuple_to_bond[local_bond_key]
                    bond_comp[idx] = [u, v]
            
            fmess.append(mess_vec)
            eid = len(edge_dict) + 1
            edge_dict[(u, v)] = eid
            agraph[v].append(eid) if v < len(agraph) else agraph.append([eid])
            while len(agraph) <= v:
                agraph.append([])
            if v < len(agraph):
                agraph[v].append(eid)
            bgraph.append([])
        
        atoms_in_bonds.extend([b for b in bond_comp if b is not None])
        
        # 构建 bgraph（前驱消息索引）
        for u, v in G.edges:
            eid = edge_dict.get((u, v))
            if eid is None:
                continue
            for w in G.predecessors(u):
                if w == v:
                    continue
                pred_eid = edge_dict.get((w, u))
                if pred_eid is not None:
                    bgraph[eid].append(pred_eid)
    
    fnode_t = torch.tensor(fnode, dtype=torch.float)
    fmess_t = torch.tensor(fmess, dtype=torch.float)
    atoms_in_bonds_t = create_pad_tensor(atoms_in_bonds)
    agraph_t = create_pad_tensor(agraph)
    bgraph_t = create_pad_tensor(bgraph)
    
    graph_tensors = (fnode_t, fmess_t, agraph_t, bgraph_t, atoms_in_bonds_t)
    return graph_tensors, (atom_scope, bond_scope)


# === 演示图张量打包 ===
print('=== Step 6: 图张量打包演示 ===')
print()

# 创建 3 个分子的图对象
smiles_list = ['CCO', 'c1ccccc1', 'CC(=O)O']
graphs = []
for smi in smiles_list:
    mol = Chem.MolFromSmiles(smi)
    for atom in mol.GetAtoms():
        atom.SetAtomMapNum(atom.GetIdx() + 1)
    graphs.append(RxnElement(mol))

tensors, scopes = pack_graph_feats_directed(graphs)
fnode, fmess, agraph, bgraph, atoms_in_bonds = tensors
atom_scope, bond_scope = scopes

print('打包后的张量形状:')
print(f'  fnode:          {fnode.shape}  (所有原子特征)')
print(f'  fmess:          {fmess.shape}  (所有消息特征)')
print(f'  agraph:         {agraph.shape}  (原子→消息索引)')
print(f'  bgraph:         {bgraph.shape}  (消息→前驱索引)')
print(f'  atoms_in_bonds: {atoms_in_bonds.shape}  (键→原子对)')
print()
print('分子作用域:')
for i, (smi, (as_start, as_len), (bs_start, bs_len)) in enumerate(
        zip(smiles_list, atom_scope, bond_scope)):
    print(f'  {smi}: 原子 [{as_start}, {as_start+as_len}), 键 [{bs_start}, {bs_start+bs_len})')

=== Step 6: 图张量打包演示 ===

打包后的张量形状:
  fnode:          torch.Size([14, 98])  (所有原子特征)
  fmess:          torch.Size([23, 8])  (所有消息特征)
  agraph:         torch.Size([14, 6])  (原子→消息索引)
  bgraph:         torch.Size([23, 2])  (消息→前驱索引)
  atoms_in_bonds: torch.Size([12, 2])  (键→原子对)

分子作用域:
  CCO: 原子 [1, 4), 键 [1, 3)
  c1ccccc1: 原子 [4, 10), 键 [3, 9)
  CC(=O)O: 原子 [10, 14), 键 [9, 12)


## 完整数据处理流程总结

| 步骤 | 输入 | 输出 | 关键函数 |
|------|------|------|----------|
| **Step 1** 产物标准化 | 原始反应 SMILES | 标准化 SMILES + 映射表 | `canonicalize_prod()` |
| **Step 2** 反应核心提取 | 反应物/产物 SMILES | 核心原子集 + 键编辑列表 | `get_reaction_core()` |
| **Step 3** 离去基解析 | 反应 SMILES + 核心 | ReactionInfo（含 lg_edits） | `get_reaction_info()` |
| **Step 4** 编辑应用 | 产物 + 编辑 | 中间体（synthons） | `apply_edits_to_mol()` |
| **Step 5** 图特征构建 | 分子 | 原子/键特征 + 邻接表 | `get_atom_features()`, `RxnElement` |
| **Step 6** 张量打包 | 图对象列表 | 批处理张量 | `pack_graph_feats()` |

### 核心数据结构

```
编辑格式:  "a1:a2:b_prod:b_reac"   (原子1:原子2:产物键序:反应物键序)
键序值:    0.0 (无键), 1.0 (单键), 1.5 (芳香键), 2.0 (双键), 3.0 (三键)

图张量:    (fnode, fmess, agraph, bgraph, atoms_in_bonds)
作用域:    (atom_scope, bond_scope)  → 定位每个分子在批处理中的位置
```

**下一步**：进入 `3_模型展示.ipynb`，学习 GraphRetro 的 GNN 编码器、键编辑预测和离去基预测模型。